In [1]:
)set message type off
)set output algebra off
setFormat!(FormatMathJax)$JFriCASSupport
)set message type on

In [2]:
)version

Value = "FriCAS d2d82c3317ba73242ae3f50891f65670f3dd3f1e compiled at Thu Oct  1 22:14:02 CEST 2020"


This notebook is licenced under [CC BY-SA 3.0]{http://creativecommons.org/licenses/by-sa/3.0/).

# FriCAS Tutorial (First Steps)

## Ralf Hemmecke <[ralf@hemmecke.org](mailto:ralf@hemmecke.org)>

## Language-defined Types
A record is a data structure with a fixed number of named entries.

In [3]:
r: Record(name: String, age: Integer) := ["Albert", 42]

Record (name: String ,age: Integer )

Getting and setting entries is done by using a dot notation.

In [4]:
r.name

String

In [5]:
r.age := 75

PositiveInteger

In [6]:
r

Record (name: String ,age: Integer )

`Union` is a data structure that can hold any value of the given types, but no value of any other type. In mathematical term, it corresponds to the disjoint union.

In [7]:
u: Union(str: String, int: Integer, flo: Float) := 4::Integer

Union (int: Integer ,...)

In [8]:
u := "some text"

Union (str: String ,...)

In [9]:
(u case int, u case str)

Tuple ( Boolean )

Tuples can be used in parallel assignments.

In [10]:
(x,y,z) := (-1,0,1)

PositiveInteger

In [11]:
(z,y,x)

Tuple ( Integer )

## Library-defined Data Types
FriCAS comes with a lot of data structures. There are lists, arrays, hash tables, trees, streams, etc.
### List
The type `List(T)` denotes linked lists whose elements all belong to type `T`.

In [12]:
li := [2,4,5,-6]

List ( Integer )

In [13]:
ls := ["I", "am", "a", "list", "of", "strings"]

List ( String )

All elements of a list must belong to the same type. This is advantageous, since the type of the list asserts something about its members.

In [14]:
concat ls

String

The following operation fails immediately without ever touching one single element of the list. Now imagine what happens in a typeless system for a list with 1000 elements that are all strings except the last one.

In [15]:
concat li

There are 2 exposed and 0 unexposed library operations named concat having 1 
argument(s) but none was determined to be applicable. Use HyperDoc Browse, or
 issue
                             )display op concat
to learn more about the available operations. Perhaps package-calling the 
operation or using coercions on the arguments will allow you to apply the 
operation.
Cannot find a definition or applicable library operation named concat with 
argument type(s) 
                                List(Integer)
 
Perhaps you should use "@" to indicate the required return type, or "$" to 
specify which version of the function you need.


FriCAS does not allow to insert an element of the wrong type into the list.

In [16]:
concat("foo", li)

There are 5 exposed and 0 unexposed library operations named concat having 2 
argument(s) but none was determined to be applicable. Use HyperDoc Browse, or
 issue
                             )display op concat
to learn more about the available operations. Perhaps package-calling the 
operation or using coercions on the arguments will allow you to apply the 
operation.
Cannot find a definition or applicable library operation named concat with 
argument type(s) 
                                   String
                                List(Integer)
 
Perhaps you should use "@" to indicate the required return type, or "$" to 
specify which version of the function you need.


In [17]:
concat(3, li)

List ( Integer )

One can access a particular element of the list with the dot notation.

In [18]:
li.3

PositiveInteger

The length of a list can be computed by the `#` operation.

In [19]:
#li

PositiveInteger

Lists can be constructed by list comprehension.

In [20]:
[3*x for x in 1..10]

List ( PositiveInteger )

The vertical bar denotes a "such-that" clause, i.e. an element only belongs to the list, if the boolean expression in the such-that clause is satisfied.

In [21]:
[3*x+1 for x in 1..20 | odd? x]

List ( PositiveInteger )

List can contain more complicated structures.
Note the parallel iteration scheme in the following expression.

In [22]:
[concat(x ,concat(y, li)) for x in 1..3 for y in -1..-100 by -2]

List ( List ( Integer ))

### Array
Arrays allow for constant time access since its elements are stored in a contiguous block of memory.

In [23]:
a := oneDimensionalArray [2,3,9,-1,-3,2,7]

OneDimensionalArray ( Integer )

In [24]:
removeDuplicates a

OneDimensionalArray ( Integer )

### Table
Hash tables allow a (nearly) constant time access. They can be thought of as a partial function from the key space to the value space.

In [25]:
upper: Table(String, String) := table()

Table ( String , String )

In [26]:
upper."a" := "A"

String

In [27]:
for i in 1..3 repeat upper("bcd".i) := "BCD".i

Void

In [28]:
upper

Table ( String , String )

In [29]:
upper."c"

String

`XHashTable` is a more efficient implementation of the table structure with (almost) linear element access time due to the use of a hash function and direct array access.
In fact, internally, `Table` is represented as an association list, i.e., uses linear time for its element access.

First clear the above variable `upper`.

In [30]:
)clear properties upper

Now we can repeat the commands from above with `Table` replaced by `XHashTable`.

In [31]:
upper: XHashTable(String, String) := table();
upper."a" := "A"
for i in 1..3 repeat upper("bcd".i) := "BCD".i
upper
upper."c"

XHashTable ( String , String )

String

Void

XHashTable ( String , String )

String

### Segment
Segmented lists provide a way to enter data efficiently.

In [32]:
sl := [2..5,3,-1,-2..5, 10..20]

List ( Segment ( Integer ))

In [33]:
expand sl

List ( Integer )

Segments can also be stored in a variable and remembered for later. With the `by` keyword one can specify a stepsize. Of course, the stepsize can be negative.

In [34]:
sg := 4 .. 20 by 3

Segment ( PositiveInteger )

In [35]:
[x^2 for x in sg | even? x]

List ( PositiveInteger )

Segments need not have an upper bound.

In [36]:
sp := 0..

UniversalSegment ( NonNegativeInteger )

Infinite data structures can be handled. A stream is like an infinite list. Elements are computed on demand.

In [37]:
even := [2*n for n in sp]

Stream ( NonNegativeInteger )

In [38]:
odd := [2*n+1 for n in 0..]

Stream ( NonNegativeInteger )

The `for` construction can be combined with a second one in order to run over two structures in parallel.

In [39]:
first9even := [n for n in even for k in 1..9]

Stream ( NonNegativeInteger )

In case the stream is finite, it can be converted into a list.

In [40]:
entries first9even

List ( NonNegativeInteger )

### Heap
The domain Heap is a special data structure that allows to insert elements in O(log n) time and extracts the maximum from the heap also in O(log n). Heaps are most appropriate for algorithms that need a priority queue.

In [41]:
h := heap  ["a", "c", "d", "b","f", "h", "z","b"]

Heap ( String )

In [42]:
[extract! h for n in 1..3]

List ( String )

In [43]:
h

Heap ( String )

In [44]:
insert!("b2",h)

Heap ( String )

In [45]:
members h

List ( String )

Because of the algorithms used to implement a heap, it makes no sense to provide a mechanism to extract the n-th element directly.

In [46]:
h.3

There are no library operations named h 
Use HyperDoc Browse or issue
                                 )what op h
to learn if there is any operation containing " h " in its name.
Cannot find a definition or applicable library operation named h with 
argument type(s) 
                               PositiveInteger
 
Perhaps you should use "@" to indicate the required return type, or "$" to 
specify which version of the function you need.


It is also impossible to create a heap over a domain that does not provide an ordering operation.

In [47]:
PrimeField 7 has OrderedSet
Heap(PrimeField 7)

Boolean

 Heap(PrimeField(7)) is not a valid type.


### Multiset
A multiset stores elements together with its occurences. It is like an unordered list with duplicates allowed.

In [48]:
li := [1,2,4,5,6,2,5,1,4,2,5]

List ( PositiveInteger )

In [49]:
multiset li

Multiset ( PositiveInteger )

### Set
A set is an unordered list with duplicated removed.

In [50]:
s := set li

Set ( PositiveInteger )

One can ask for the size of the set.

In [51]:
#s

PositiveInteger

Apply the usual set operations like union, intersection, set difference, etc.

In [52]:
union(s, set [3,4,5])
intersect(s, set [3,4,5])

Set ( PositiveInteger )

Set ( PositiveInteger )

Since a set is unordered, there is no concept of a k-th element.

In [53]:
s.3

There are no library operations named s 
Use HyperDoc Browse or issue
                                 )what op s
to learn if there is any operation containing " s " in its name.
Cannot find a definition or applicable library operation named s with 
argument type(s) 
                               PositiveInteger
 
Perhaps you should use "@" to indicate the required return type, or "$" to 
specify which version of the function you need.


### Stack
A stack implements a LIFO queue (last-in, first-out). It is like a list, but not all list operations are available. For example, one cannot remove an element which is not the top-most element.

In [54]:
s := stack [2,4,-2,3,6]

Stack ( Integer )

In [55]:
top s

PositiveInteger

In [56]:
push!(-7,s)

Integer

In [57]:
s

Stack ( Integer )

In [58]:
#s

PositiveInteger

In [59]:
pop! s

Integer

In [60]:
s

Stack ( Integer )

There are many more data structures. Among them are trees and queues.

In [61]:
)what domain Tree

--------------------------------- Domains ---------------------------------
Domains with names matching patterns:
     tree 
 BBTREE   BalancedBinaryTree           BSTREE   BinarySearchTree
 BTCAT-   BinaryTreeCategory&          BTREE    BinaryTree
 PENDTREE PendantTree                  SPLTREE  SplittingTree
 TREE     Tree


In [62]:
)what domain queue

--------------------------------- Domains ---------------------------------
Domains with names matching patterns:
     queue 
 DEQUEUE  Dequeue                      QUEUE    Queue


In [63]:
)what domain Stream

--------------------------------- Domains ---------------------------------
Domains with names matching patterns:
     stream 
 BITST    BitStreamFrame               LZSTAGG- LazyStreamAggregate&
 STAGG-   StreamAggregate&             STREAM   Stream
